In [1]:
import os

SEASONS = list(range(2016,2026))

DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR, "standings")
SCORES_DIR = os.path.join(DATA_DIR, "scores")

In [2]:
SEASONS

[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [3]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time

In [4]:

async def get_html(url, selector, sleep=5, retries=3):
    html = None
    for i in range(1, retries+1):
        time.sleep(sleep * i)
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)
        except PlaywrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break
    return html

In [5]:
async def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = await get_html(url, "#content .filter")
    
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    standings_pages = [f"https://www.basketball-reference.com{l['href']}" for l in links]
    
    for url in standings_pages:
        save_path = os.path.join(STANDINGS_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        
        html = await get_html(url, "#all_schedule")
        with open(save_path, "w+") as f:
            f.write(html)

In [6]:
for season in SEASONS:
    await scrape_season(season)
    

2015-16 NBA Schedule | Basketball-Reference.com
2016-17 NBA Schedule | Basketball-Reference.com
2017-18 NBA Schedule | Basketball-Reference.com
2018-19 NBA Schedule | Basketball-Reference.com
2019-20 NBA Schedule | Basketball-Reference.com
2020-21 NBA Schedule | Basketball-Reference.com
2021-22 NBA Schedule | Basketball-Reference.com
2022-23 NBA Schedule | Basketball-Reference.com
2023-24 NBA Schedule | Basketball-Reference.com
Timeout error on https://www.basketball-reference.com/leagues/NBA_2025_games.html
2024-25 NBA Schedule | Basketball-Reference.com


In [7]:
standings_files = os.listdir(STANDINGS_DIR)

In [8]:
async def scrape_game(standings_file):
    with open(standings_file, 'r') as f:
        html = f.read()

    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get('href') for l in links]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in hrefs if l and "boxscore" in l and '.html' in l]

    for url in box_scores:
        save_path = os.path.join(SCORES_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue

        html = await get_html(url, "#content")
        if not html:
            continue
        with open(save_path, "w+") as f:
            f.write(html)
            

In [9]:
import pandas as pd

for season in SEASONS:
    files = [s for s in standings_files if str(season) in s]
    
    for f in files:
        filepath = os.path.join(STANDINGS_DIR, f)
        
        await scrape_game(filepath)
    

Pistons vs Pacers, February 6, 2016 | Basketball-Reference.com
Pelicans vs Cavaliers, February 6, 2016 | Basketball-Reference.com
Nets vs 76ers, February 6, 2016 | Basketball-Reference.com
Mavericks vs Grizzlies, February 6, 2016 | Basketball-Reference.com
Bulls vs Timberwolves, February 6, 2016 | Basketball-Reference.com
Lakers vs Spurs, February 6, 2016 | Basketball-Reference.com
Thunder vs Warriors, February 6, 2016 | Basketball-Reference.com
Jazz vs Suns, February 6, 2016 | Basketball-Reference.com
Kings vs Celtics, February 7, 2016 | Basketball-Reference.com
Nuggets vs Knicks, February 7, 2016 | Basketball-Reference.com
Hawks vs Magic, February 7, 2016 | Basketball-Reference.com
Clippers vs Heat, February 7, 2016 | Basketball-Reference.com
Bulls vs Hornets, February 8, 2016 | Basketball-Reference.com
Kings vs Cavaliers, February 8, 2016 | Basketball-Reference.com
Lakers vs Pacers, February 8, 2016 | Basketball-Reference.com
Timeout error on https://www.basketball-reference.com/box

Error: Page.goto: net::ERR_NETWORK_IO_SUSPENDED at https://www.basketball-reference.com/boxscores/201511270ORL.html
Call log:
  - navigating to "https://www.basketball-reference.com/boxscores/201511270ORL.html", waiting until "load"
